In [1]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import random


In [ ]:

# Define dataset class
class GestureDataset(Dataset):
    def __init__(self, data_path, batch_size=32):
        self.data = []
        self.labels = []
        self.batch_size = batch_size
        self.classes = sorted(os.listdir(data_path))
        self.class_to_idx = {cls: idx for idx, cls in enumerate(self.classes)}

        for cls in self.classes:
            class_path = os.path.join(data_path, cls)
            if os.path.isdir(class_path):
                for file in os.listdir(class_path):
                    if file.endswith('.csv'):
                        file_path = os.path.join(class_path, file)
                        sequence = np.loadtxt(file_path, delimiter=',', dtype=np.float32)
                        self.data.append(torch.tensor(sequence, device=device))  # Load directly onto GPU
                        self.labels.append(torch.tensor(self.class_to_idx[cls], device=device))  # Labels also on GPU

        print(f"Loaded {len(self.data)} samples from {data_path}")
        self.balance_batches()

    def balance_batches(self):
        remainder = len(self.data) % self.batch_size
        if remainder > 0:
            extra_samples = self.batch_size - remainder
            indices = np.random.choice(len(self.data), extra_samples, replace=True)
            self.data.extend([self.data[i] for i in indices])
            self.labels.extend([self.labels[i] for i in indices])

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]

# Collate function for padding
def collate_fn(batch):
    sequences, labels = zip(*batch)
    sequences = nn.utils.rnn.pad_sequence(sequences, batch_first=True, padding_value=0.0)
    labels = torch.stack(labels)
    return sequences, labels

# Define LSTM model
class LSTMGestureModel(nn.Module):
    def __init__(self, input_dim=63, hidden_dim=256, output_dim=128, num_layers=2):
        super(LSTMGestureModel, self).__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x, _ = self.lstm(x)
        return self.fc(x[:, -1, :])  # Take last time step output

# Define contrastive loss
class ContrastiveLoss(nn.Module):
    def __init__(self, margin=1.0):
        super(ContrastiveLoss, self).__init__()
        self.margin = margin
    
    def forward(self, output1, output2, target):
        euclidean_distance = torch.norm(output1 - output2, dim=1)
        loss = target * euclidean_distance.pow(2) + (1 - target) * F.relu(self.margin - euclidean_distance).pow(2)
        return loss.mean()

# Training setup
data_path = 'augmented_train_data_concatenated_delta_landmark0'
test_data_path = 'augmented_test_data_concatenated_delta_landmark0'
batch_size = 32
num_epochs = 20
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

train_dataset = GestureDataset(data_path, batch_size=batch_size)
test_dataset = GestureDataset(test_data_path, batch_size=batch_size)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)

model = LSTMGestureModel().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)
loss_fn = ContrastiveLoss().to(device)


# Training loop
def train():
    for epoch in range(1, num_epochs + 1):
        model.train()
        total_loss = 0.0
        data_count = 0
        
        for data1, labels1 in train_loader:
            for data2, labels2 in train_loader:
                if torch.equal(data1, data2):
                    continue
                
                target = (labels1 == labels2).float()
                
                output1 = model(data1)
                output2 = model(data2)
                loss = loss_fn(output1, output2, target)
                
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                
                total_loss += loss.item()
                data_count += 1

        avg_loss = total_loss / data_count
        print(f"Epoch [{epoch}/{num_epochs}], Loss: {avg_loss:.4f}")
        
        # Evaluate on test data
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for data1, labels1 in test_loader:
                for data2, labels2 in test_loader:
                    target = (labels1 == labels2).float()
                    
                    output1 = model(data1)
                    output2 = model(data2)
                    euclidean_distance = torch.norm(output1 - output2, dim=1)
                    predictions = (euclidean_distance < 1.0) == target.byte()
                    correct += predictions.sum().item()
                    total += labels1.size(0)
        
        accuracy = correct / total
        print(f"Test Accuracy: {accuracy:.4f}")
        
        # Save model
        torch.save(model.state_dict(), f'model_epoch_{epoch}.pth')
        print(f"Model saved at epoch {epoch}")




train()
print("Training complete.")


Using device: cuda
Loaded 1564 samples from augmented_train_data_concatenated_delta_landmark0
Loaded 1046 samples from augmented_test_data_concatenated_delta_landmark0


In [8]:
'hi'

'hi'